In [1]:
import pandas as pd
import numpy as np
import re
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import quote_plus

/disk1/anaconda3/envs/kcat/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
df = pd.read_csv('data/sabiork_dataset.csv')
df.head()

,Enzyme Variant,ECNumber,UniProtKB_AC,Organism,Temperature,pH,parameter.name,parameter.type,parameter.associatedSpecies,parameter.startValue,parameter.endValue,parameter.standardDeviation,parameter.unit,Substrate
0,mutant MDH G81S,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,S,concentration,(S)-Mandelate,NaN,NaN,-,-,Riboflavin-5-phosphate;(S)-Mandelate
1,mutant MDH G81S,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,kcat,kcat,NaN,2.80000,NaN,1E-1,s^(-1),Riboflavin-5-phosphate;(S)-Mandelate
2,mutant MDH G81S,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,Km,Km,(S)-Mandelate,0.00230,NaN,2E-4,M,Riboflavin-5-phosphate;(S)-Mandelate
3,mutant MDH G81V,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,Km,Km,(S)-Mandelate,0.00017,NaN,6E-5,M,Riboflavin-5-phosphate;(S)-Mandelate
4,mutant MDH G81V,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,S,concentration,(S)-Mandelate,NaN,NaN,-,-,Riboflavin-5-phosphate;(S)-Mandelate


In [ ]:
df = df.rename(columns={'ECNumber': 'ec_number', 'Substrate': 'substrate', 'UniProtKB_AC': 'uniprot',
                        'Organism': 'organism', 'Temperature': 'temperature'})

In [4]:
df.columns

Index(['Enzyme Variant', 'ec_number', 'uniprot', 'organism', 'temperature',
       'pH', 'parameter.name', 'parameter.type', 'parameter.associatedSpecies',
       'parameter.startValue', 'parameter.endValue',
       'parameter.standardDeviation', 'parameter.unit', 'substrate'],
      dtype='object')

In [5]:
df['parameter.type'].unique()

array(['concentration', 'kcat', 'Km', 'kcat/Km', 'Vmax',
       'Hill coefficient', 'S_half', 'specific enz. activity', 'Ki',
       'pKa', 'pH', 'Kd', 'rate const.', 'unknown', 'Hill constant',
       'Keq', 'Vmax/Km', 'IC50', 'kinact', 'EC50', 'Activation constant',
       'specific concentration', 'time', 'kcat/S_half', 'half-life',
       'enthalpy change'], dtype=object)

In [ ]:
df = df[df['parameter.type'] == 'kcat']

print(f"After filtering: {len(df)}")

필터링 후 데이터 개수: 23521


In [ ]:
df = df.dropna(subset=['pH', 'temperature'])
print(f"After filtering: {len(df)}")

필터링 후 데이터 개수: 23521


## Extract sequence with Uniprot ID

In [ ]:
df = df[~df['uniprot'].str.contains(';', na=False)]
print('Data with one Uniprot ID: ', len(df))

Uniprot 값이 한 개만 있는 데이터:  22500


In [9]:
uniprot_ids = list(df['uniprot'].unique())
len(uniprot_ids)

2418

In [ ]:
def get_protein_sequence(uniprot_id: str):
    base_url = "https://rest.uniprot.org/uniprotkb/"
    url = f"{base_url}{uniprot_id}.fasta"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise exception for HTTP errors
        
        # Parse the FASTA format
        lines = response.text.split('\n')
        if len(lines) > 1:
            # Join all lines except the first one (header)
            sequence = ''.join(lines[1:])
            return sequence
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching sequence for {uniprot_id}: {e}")
        return None

In [9]:
sequences = []

for id in tqdm(uniprot_ids):
    try:
        seq = get_protein_sequence(id)
        sequences.append(seq)
    except:
        sequences.append(None)

uniprot_db = pd.DataFrame({'uniprot': uniprot_ids, 'sequence': sequences})
uniprot_db.to_csv('data/sabiork_uniprot_db.csv', index=False)

  0%|          | 0/2418 [00:00<?, ?it/s]

  0%|          | 7/2418 [00:07<43:34,  1.08s/it]

Error fetching sequence for nan: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/nan.fasta


100%|██████████| 2418/2418 [44:25<00:00,  1.10s/it]


In [8]:
uniprot_db = pd.read_csv('data/sabiork_uniprot_db.csv')

In [9]:
df = df.merge(uniprot_db[['uniprot', 'sequence']], on='uniprot', how='left')

In [ ]:
df = df.dropna(subset=['sequence'])
print(f"After removing data with no sequence information: {len(df)}")

Sequence가 없는 데이터 제거 후 데이터 개수: 21286


## Add wildtype/mutant information

In [ ]:
def extract_type_and_mutation(comment):
    if pd.isna(comment):
        return None, None
    
    is_wildtype = any(word in comment.lower() for word in ['wildtype', 'wild-type', 'wild type'])
    is_mutant = 'mutant' in comment.lower()
    
    mutations = re.findall(r'([A-Z]\d+[A-Z])', comment)
    
    if is_wildtype:
        return 'wildtype', None
    elif is_mutant:
        if mutations:
            return 'mutant', '/'.join(mutations)
        else:
            return 'mutant', None
    else:
        return None, None

df['type'] = None
df['mutation'] = None

for idx, row in df.iterrows():
    type_val, mutation_val = extract_type_and_mutation(row['Enzyme Variant'])
    df.at[idx, 'type'] = type_val
    df.at[idx, 'mutation'] = mutation_val

In [ ]:
no_type_info = df[df['type'].isna()]
print(f"Data with no type information: {len(no_type_info)}")

df = df[~((df['type'] == 'mutant') & (df['mutation'].isna()))]

print("\nTotal data length: ", len(df))
print(f"Wildtype data length: {len(df[df['type'] == 'wildtype'])}")
print(f"Mutant data length: {len(df[df['type'] == 'mutant'])}")

타입 정보가 없는 데이터 수: 0

총 데이터 수:  20275
Wildtype 데이터 수: 11783
Mutant 데이터 수: 8492


## Preprocess turnover number

In [13]:
df['turnover_number'] = df['parameter.startValue']

In [14]:
df['parameter.unit'].unique()

array(['s^(-1)', 'M^(-1)*s^(-1)', '-', 'M', 'mol*s^(-1)*mol^(-1)',
       'katal', 'J/mol', 'mol*s^(-1)*g^(-1)'], dtype=object)

In [15]:
df['parameter.name'].unique()

array(['kcat', 'k_cat', 'kcat_max', 'Kcat', 'kmax', 'kcat_2', 'kcat_1',
       'kcat_c', 'kcat_b', 'kcat_a', 'kcat0', 'kcat_B', 'kcat2', 'kcat1',
       'Kcat0', 'kcatA', 'kcatB', 'kcatR', 'kcatF', 'kcatE', 'kcat_E',
       'kcat_R', 'kcat_A', 'k0', 'V/Et', 'kc', 'k1', 'k2', 'kcatG',
       'kcat_0', 'kcat_C', 'kcat_D', 'kcat_r', 'kcat_f', 'kobs', 'v_dec',
       'kcat_act', 'kact', 'kcat_app', 'Vf_ET', 'Kcat1', 'Kcat2'],
      dtype=object)

In [ ]:
valid_kcat_units = {
    's^(-1)': 's^(-1)',
    '1/s': 's^(-1)',
    'mol*s^(-1)*mol^(-1)': 's^(-1)',  # mol/mol/s → s⁻¹
}

df = df[df['parameter.unit'].isin(valid_kcat_units)]

df['parameter.unit'] = df['parameter.unit'].map(valid_kcat_units)

In [ ]:
df = df.dropna(subset=['turnover_number'])
print('After preprocessing turnover number data: ', len(df))

Turnover number 데이터 정제 후 데이터 개수:  20241


In [18]:
df.isna().sum()

Enzyme Variant                     0
ec_number                          0
uniprot                            0
organism                           0
temperature                        0
pH                                 0
parameter.name                     0
parameter.type                     0
parameter.associatedSpecies    20241
parameter.startValue               0
parameter.endValue             20161
parameter.standardDeviation        0
parameter.unit                     0
substrate                          0
sequence                           0
type                               0
mutation                       11768
turnover_number                    0
dtype: int64

In [ ]:
sabio_substrates = ''
for i in range(len(df)):
    sabio_substrates += df.iloc[i]['substrate'] + ';'

sabio_substrates = list(set(sabio_substrates.split(';')))
print(len(sabio_substrates))

2861


In [21]:
missing_in_substrates = set(sabio_substrates) - set(substrates['substrate'])
missing_in_substrates = list(missing_in_substrates)
missing_in_substrates

['',
 '4-Hydroxybenzaldehyde',
 '(E)-2-Methylgeranyl diphosphate',
 'Neomycin B',
 '12(13)-EpOME',
 '4-Fluorobenzoyl-CoA',
 'Leu-enkephalin',
 'Acyclovir',
 'Nicotinamide nucleotide',
 'Maltotetraose',
 'Delphinidin',
 'UDPMurAc(oyl-L-Ala-D-gamma-Glu-L-Lys-D-Ala-D-Ala)',
 'L-2,4-Diaminobutanoate',
 'Hydrogen sulfide',
 'cis-Aconitate',
 '1-Myristoylglycerol 3-phosphate',
 '(S)-Reticuline',
 '(2E)-2-Hydroxypenta-2,4-dienoic acid',
 '(3R)-3-Fluoro-3-deoxy-D-arabino-heptulosonate 7-phosphate',
 'RNA',
 'Lacto-N-tetraose',
 '3-Formylcatechol',
 '1,4-beta-D-Mannan',
 'alpha-Casein',
 'Laminaribiose',
 'L-2-Aminoadipate',
 'Aniline',
 't-Butyloxycarbonyl-Phe-Ser-Arg-4-methylcoumarin-7-ylamide',
 'N-Methyldihydronicotinamide',
 '2-Hydroxycaproate',
 'Starch',
 '12-Hydroxydodecanoic acid',
 'L-Kynurenine',
 'R-(-)-3-Methylbutan-2-ol',
 'Glutathione ethyl ester-methylglyoxal hemithioacetal',
 'Linoleate',
 'Cyanamide',
 'O-Acetyl-ADP-ribose',
 'Cephalexin',
 'Boldenone',
 '3-O-Methyl-D-glucose'

In [ ]:
def get_cid_from_google(substrate):
    try:
        # PubChem PUG REST API URL
        search_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quote_plus(substrate)}/cids/JSON"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(search_url, headers=headers)
        
        if response.status_code != 200:
            print(f"Failed to get response for {substrate}")
            return None
            
        data = response.json()
        if 'IdentifierList' in data and 'CID' in data['IdentifierList']:
            cids = data['IdentifierList']['CID']
            if cids:
                return str(cids[0]) 
                
        print(f"Could not find CID in PubChem for {substrate}")
        return None
        
    except Exception as e:
        print(f"Error getting CID from PubChem for {substrate}: {str(e)}")
        return None

def get_smiles(cid):
    try:
        # PubChem PUG REST API URL
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSMILES,CanonicalSMILES/JSON"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return None, None
            
        data = response.json()
        if 'PropertyTable' in data and 'Properties' in data['PropertyTable']:
            properties = data['PropertyTable']['Properties'][0]
            return properties.get('IsomericSMILES'), properties.get('CanonicalSMILES')
            
        return None, None
        
    except Exception as e:
        print(f"Error getting SMILES for CID {cid}: {str(e)}")
        return None, None

In [ ]:
for substrate in missing_in_substrates:
    try:
        cid = get_cid_from_google(substrate)
        
        if cid is not None:
            isomeric_smiles, canonical_smiles = get_smiles(cid)

            if isomeric_smiles is not None:
                new_row = {
                    'substrate': substrate,
                    'isomeric_smiles': isomeric_smiles,
                    'canonical_smiles': canonical_smiles
                }
                substrates = pd.concat([substrates, pd.DataFrame([new_row])], ignore_index=True)
                print(f"Added: {substrate}")
    except:
        print(f"Error ({substrate}): {e}")

print(f"Final substrate length: {len(substrates)}")

Failed to get response for 
Failed to get response for (E)-2-Methylgeranyl diphosphate
Failed to get response for Neomycin B
Failed to get response for Nicotinamide nucleotide
Failed to get response for Hydrogen sulfide
Failed to get response for 1-Myristoylglycerol 3-phosphate
Failed to get response for (2E)-2-Hydroxypenta-2,4-dienoic acid
Failed to get response for (3R)-3-Fluoro-3-deoxy-D-arabino-heptulosonate 7-phosphate
Failed to get response for RNA
Failed to get response for 3-Formylcatechol
Failed to get response for alpha-Casein
Failed to get response for t-Butyloxycarbonyl-Phe-Ser-Arg-4-methylcoumarin-7-ylamide
Failed to get response for N-Methyldihydronicotinamide
Failed to get response for Starch
Failed to get response for 12-Hydroxydodecanoic acid
Failed to get response for R-(-)-3-Methylbutan-2-ol
Failed to get response for Glutathione ethyl ester-methylglyoxal hemithioacetal
Failed to get response for (S)-alpha-Cyano(6-methoxy-2-naphthyl)-methyl-(2S)-2-(4-chlorophenyl)-3-

In [ ]:
def process_substrate_smiles(substrate):
    substrate_parts = [s.strip() for s in substrate.split(";")]
    
    isomeric_smiles = []
    canonical_smiles = []
    for part in substrate_parts:
        matching_row = substrates[substrates['substrate'] == part]
        
        if not matching_row.empty:
            isomeric = matching_row.iloc[0]['isomeric_smiles']
            canonical = matching_row.iloc[0]['canonical_smiles']
            
            if isomeric is not None:
                isomeric_smiles.append(isomeric)
                canonical_smiles.append(canonical)
        else:
            isomeric_smiles = []
            canonical_smiles = []

    try:
        isomeric_smiles = ";".join(isomeric_smiles)
        canonical_smiles = ";".join(canonical_smiles)
        return isomeric_smiles, canonical_smiles
    except:
        return None, None

In [26]:
isomeric_smiles, canonical_smiles = [], []

df = df.reset_index(drop=True)

for i in range(len(df)):
    isomeric, canonical = process_substrate_smiles(df.loc[i, 'substrate'])
    isomeric_smiles.append(isomeric)
    canonical_smiles.append(canonical)

df['isomeric_smiles'] = isomeric_smiles
df['canonical_smiles'] = canonical_smiles

In [ ]:
print('Data with no substrate SMILES: ', len(df[df['isomeric_smiles'].isna()]))

Substrate SMILES가 없는 데이터 수:  86


## Add mutation information to sequence

In [ ]:
def apply_mutations(sequence, mutation_info):
    mutations = mutation_info.split('/')
    seq_len = len(sequence)

    for mutation in mutations:
        wt_aa = mutation[0]
        mt_aa = mutation[-1]
        pos = int(mutation[1:-1])-1 # Python number

        try:
            if sequence[pos] == wt_aa:
                sequence = sequence[:pos] + mt_aa + sequence[pos+1:]
            elif sequence[pos+1] == wt_aa:
                sequence = sequence[:pos+1] + mt_aa + sequence[pos+2:]
            elif sequence[pos+2] == wt_aa:
                sequence = sequence[:pos+2] + mt_aa + sequence[pos+3:]
            else:
                print(f'Pos {pos} does not match wildtype amino acid {wt_aa}, {sequence[pos]}')
                print(f"Mutation {mutation} does not match wildtype amino acid {wt_aa} at position {pos}")
                print(f"Sequence: {sequence}")
                return None
        except:
            print(f"Mutation {mutation} is out of sequence length | sequence length: {seq_len}")
            return None

    if len(sequence) != seq_len:
        print(f"Sequence length changed from {seq_len} to {len(sequence)}")
        print(f"Sequence: {sequence}")
        return None
    
    return sequence

In [29]:
df = df.reset_index(drop=True)

In [30]:
df.head()

,Enzyme Variant,ec_number,uniprot,organism,temperature,pH,parameter.name,parameter.type,parameter.associatedSpecies,parameter.startValue,parameter.endValue,parameter.standardDeviation,parameter.unit,substrate,sequence,type,mutation,turnover_number,isomeric_smiles,canonical_smiles
0,mutant MDH G81S,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,kcat,kcat,NaN,2.80,NaN,1E-1,s^(-1),Riboflavin-5-phosphate;(S)-Mandelate,MSQNLFNVEDYRKLRQKRLPKMVYDYLEGGAEDEYGVKHNRDVFQQ...,mutant,G81S,2.80,C1=CC=C(C=C1)[C@@H](C(=O)[O-])O,C1=CC=C(C=C1)C(C(=O)[O-])O
1,mutant MDH G81V,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,kcat,kcat,NaN,0.05,NaN,4E-3,s^(-1),Riboflavin-5-phosphate;(S)-Mandelate,MSQNLFNVEDYRKLRQKRLPKMVYDYLEGGAEDEYGVKHNRDVFQQ...,mutant,G81V,0.05,C1=CC=C(C=C1)[C@@H](C(=O)[O-])O,C1=CC=C(C=C1)C(C(=O)[O-])O
2,mutant MDH G81D,1.1.99.31,P20932,Pseudomonas putida,20.0,7.5,kcat,kcat,NaN,0.11,NaN,4E-3,s^(-1),Riboflavin-5-phosphate;(S)-Mandelate,MSQNLFNVEDYRKLRQKRLPKMVYDYLEGGAEDEYGVKHNRDVFQQ...,mutant,G81D,0.11,C1=CC=C(C=C1)[C@@H](C(=O)[O-])O,C1=CC=C(C=C1)C(C(=O)[O-])O
3,wildtype DHFR,1.5.1.3,P0ABQ4,Escherichia coli,15.0,7.0,k_cat,kcat,NaN,6.18,NaN,-,s^(-1),NADPH;Dihydrofolate;H+,MISLIAALAVDRVIGMENAMPWNLPADLAWFKRNTLNKPVIMGRHT...,wildtype,None,6.18,,
4,mutant DHFR W30A,1.5.1.3,P0ABQ4,Escherichia coli,15.0,7.0,k_cat,kcat,NaN,14.50,NaN,-,s^(-1),NADPH;Dihydrofolate;H+,MISLIAALAVDRVIGMENAMPWNLPADLAWFKRNTLNKPVIMGRHT...,mutant,W30A,14.50,,


In [31]:
sequences = []

df = df.dropna(subset=['sequence']).reset_index(drop=True)

for i in range(len(df)):
    if pd.notna(df.loc[i, 'mutation']):
        seq = apply_mutations(df.iloc[i]['sequence'], df.iloc[i]['mutation'])
        sequences.append(seq)
    else:
        sequences.append(df.loc[i, 'sequence'])

Pos 1698 does not match wildtype amino acid K, V
Mutation K1699A does not match wildtype amino acid K at position 1698
Sequence: MEEVVIAGMSGKLPESENLQEFWDNLIGGVDMVTDDDRRWKAGLYGLPRRSGKLKDLSRFDASFFGVHPKQAHTMDPQLRLLLEVTYEAIVDGGINPDSLRGTHTGVWVGVSGSETSEALSRDPETLVGYSMVGCQRAMMANRLSFFFDFRGPSIALDTACSSSLMALQNAYQAIHSGQCPAAIVGGINVLLKPNTSVQFLRLGMLSPEGTCKAFDTAGNGYCRSEGVVAVLLTKKSLARRVYATILNAGTNTDGFKEQGVTFPSGDIQEQLIRSLYQSAGVAPESFEYIEAHGTGTKVGDPQELNGITRALCATRQEPLLIGSTKSNMGHPEPASGLAALAKVLLSLEHGLWAPNLHFHSPNPEIPALLDGRLQVVDQPLPVRGGNVGINSFGFGGSNVHIILRPNTQPPPAPAPHATLPRLLRASGRTPEAVQKLLEQGLRHSQDLAFLSMLNDIAAVPATAMPFRGYAVLGGERGGPEVQQVPAGERPLWFICSGMGTQWRGMGLSLMRLDRFRDSILRSDEAVKPFGLKVSQLLLSTDESTFDDIVHSFVSLTAIQIGLIDLLSCMGLRPDGIVGHSLGEVACGYADGCLSQEEAVLAAYWRGQCIKEAHLPPGAMAAVGLSWEECKQRCPPGVVPACHNSKDTVTISGPQAPVFEFVEQLRKEGVFAKEVRTGGMAFHSYFMEAIAPPLLQELKKVIREPKPRSARWLSTSIPEAQWHSSLARTSSAEYNVNNLVSPVLFQEALWHVPEHAVVLEIAPHALLQAVLKRGLKPSCTIIPLMKKDHRDNLEFFLAGIGRLHLSGIDANPNALFPPVEFPAPRGTPLISPLIKWDHSLAWDVPAAEDFPNGSGSPSAAIYNIDTSSESP

In [32]:
df['sequence'] = sequences

In [ ]:
print('Sequence with unmatched mutation information: ', len(df[df['sequence'].isna()]))
df = df.dropna(subset=['sequence'])
print('df length: ', len(df))

Sequence 정보가 틀린 데이터 수:  1636
df 길이:  18605


In [34]:
df.columns

Index(['Enzyme Variant', 'ec_number', 'uniprot', 'organism', 'temperature',
       'pH', 'parameter.name', 'parameter.type', 'parameter.associatedSpecies',
       'parameter.startValue', 'parameter.endValue',
       'parameter.standardDeviation', 'parameter.unit', 'substrate',
       'sequence', 'type', 'mutation', 'turnover_number', 'isomeric_smiles',
       'canonical_smiles'],
      dtype='object')

In [35]:
df.to_csv('data/sabiork_sequence_substrate.csv', index=False)